# RAG Retrieval for Osteosarcoma Clinical Trials

This notebook builds a retrieval pipeline over the 30-trial osteosarcoma corpus.

The workflow is:

1. Load the 30-trial dataset
2. Prepare one retrieval document per trial
3. Build a TF-IDF vector index
4. Convert synthetic patient profiles into text queries
5. Retrieve the most relevant clinical trials
6. Inspect retrieval quality
7. Later connect retrieved trials to symbolic eligibility reasoning

In [2]:
%pip install scikit-learn

   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.2 MB 4.2 MB/s eta 0:00:02
   -------- ------------------------------- 1.8/8.2 MB 4.7 MB/s eta 0:00:02
   ---------------- ----------------------- 3.4/8.2 MB 5.7 MB/s eta 0:00:01
   ------------------------ --------------- 5.0/8.2 MB 6.8 MB/s eta 0:00:01
   -------------------------------------- - 7.9/8.2 MB 7.9 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 8.0 MB/s  0:00:01
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   -- ------------------------------------- 2.6/36.6 MB 14.2 MB/s eta 0:00:03
   ------- -------------------------------- 6.6/36.6 MB 16.0 MB/s eta 0:00:02
   ------------ --------------------------- 11.0/36.6 MB 17.8 MB/s eta 0:00:02
   ---------------- ----------------------- 14.9/36.6 MB 17.9 MB/s eta 0:00:02
   -------------------- ------------------- 19.1/36.6 MB 18.4 MB/s eta 0:00:01
   ------------

In [3]:
import json
from pathlib import Path

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
PROJECT_ROOT = Path("..")

TRIALS_DIR = PROJECT_ROOT / "data" / "trials"
PATIENTS_DIR = PROJECT_ROOT / "data" / "synthetic_patients"
RULES_DIR = PROJECT_ROOT / "data" / "rules"
RAG_DIR = PROJECT_ROOT / "data" / "rag"

RAG_DIR.mkdir(parents=True, exist_ok=True)

print("Trials:", TRIALS_DIR.resolve())
print("Patients:", PATIENTS_DIR.resolve())
print("Rules:", RULES_DIR.resolve())
print("RAG:", RAG_DIR.resolve())

Trials: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\trials
Patients: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\synthetic_patients
Rules: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\rules
RAG: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\rag


In [5]:
trials_30 = pd.read_csv(
    TRIALS_DIR / "all_30_trials.csv"
)

print("Number of trials:", len(trials_30))
print("\nColumns:")
print(trials_30.columns.tolist())

Number of trials: 30

Columns:
['Review Rank', 'NCT Number', 'Study Title', 'Study Status', 'Conditions', 'Age Categories', 'Study Type', 'Phase', 'Interventions', 'Last Update Posted', 'Study URL', 'Review Notes', 'Eligibility Text Retrieved', 'Minimum Age', 'Maximum Age', 'Full Eligibility Criteria']


In [6]:
trials_30[
    [
        "NCT Number",
        "Study Title",
        "Study Status",
        "Conditions",
        "Minimum Age",
        "Maximum Age",
        "Interventions"
    ]
].head()

,NCT Number,Study Title,Study Status,Conditions,Minimum Age,Maximum Age,Interventions
0,NCT04616560,Trastuzumab Deruxtecan (DS-8201a) for the Trea...,RECRUITING,Desmoplastic Small Round Cell Tumor|Osteosarco...,12 Years,39 Years,PROCEDURE: Biospecimen Collection|PROCEDURE: C...
1,NCT05235165,Thoracotomy Versus Thoracoscopic Management of...,RECRUITING,Metastatic Malignant Neoplasm in the Lung|Meta...,NaN,50 Years,PROCEDURE: Biospecimen Collection|PROCEDURE: C...
2,NCT06492954,Atezolizumab in Combination With Stereotactic ...,RECRUITING,Osteosarcoma|Pulmonary Recurrence of Osteosarcoma,NaN,50 Years,PROCEDURE: Surgical Resection|RADIATION: Stere...
3,NCT06892678,DFMO Maintenance for Patients With Relapsed/Re...,RECRUITING,Osteosarcoma Recurrent|Ewing's Tumor Recurrent,NaN,39 Years,DRUG: DFMO
4,NCT07144254,Tegavivint With Gemcitabine in Patients With R...,RECRUITING,Osteosarcoma Recurrent|Osteosarcoma in Childre...,1 Year,30 Years,DRUG: Tegavivint|DRUG: Gemcitabine


In [7]:
print(
    trials_30["Full Eligibility Criteria"]
    .isna()
    .value_counts()
)

Full Eligibility Criteria
False    30
Name: count, dtype: int64


In [8]:
missing_eligibility = trials_30[
    trials_30["Full Eligibility Criteria"].isna()
]

print(
    "Trials missing full eligibility criteria:",
    len(missing_eligibility)
)

Trials missing full eligibility criteria: 0


In [9]:
def clean_text(value):
    if pd.isna(value):
        return ""

    return str(value).replace("\n", " ").strip()

In [10]:
def build_trial_document(row):

    parts = [
        f"NCT ID: {clean_text(row['NCT Number'])}",
        f"Title: {clean_text(row['Study Title'])}",
        f"Status: {clean_text(row['Study Status'])}",
        f"Conditions: {clean_text(row['Conditions'])}",
        f"Age categories: {clean_text(row['Age Categories'])}",
        f"Minimum age: {clean_text(row['Minimum Age'])}",
        f"Maximum age: {clean_text(row['Maximum Age'])}",
        f"Study type: {clean_text(row['Study Type'])}",
        f"Phase: {clean_text(row['Phase'])}",
        f"Interventions: {clean_text(row['Interventions'])}",
        f"Eligibility criteria: {clean_text(row['Full Eligibility Criteria'])}"
    ]

    return " ".join(parts)

In [11]:
trials_30["retrieval_document"] = trials_30.apply(
    build_trial_document,
    axis=1
)

print(
    trials_30.loc[0, "retrieval_document"][:1500]
)

NCT ID: NCT04616560 Title: Trastuzumab Deruxtecan (DS-8201a) for the Treatment of Recurrent or Refractory Osteosarcoma, Wilms Tumor, and Desmoplastic Small Round Cell Tumor Status: RECRUITING Conditions: Desmoplastic Small Round Cell Tumor|Osteosarcoma|Recurrent Desmoplastic Small Round Cell Tumor|Recurrent Kidney Wilms Tumor|Recurrent Osteosarcoma|Refractory Desmoplastic Small Round Cell Tumor|Refractory Wilms Tumor|Wilms Tumor Age categories: CHILD, ADULT Minimum age: 12 Years Maximum age: 39 Years Study type: INTERVENTIONAL Phase: PHASE2 Interventions: PROCEDURE: Biospecimen Collection|PROCEDURE: Computed Tomography|PROCEDURE: Echocardiography Test|PROCEDURE: Magnetic Resonance Imaging|PROCEDURE: Multigated Acquisition Scan|BIOLOGICAL: Trastuzumab Deruxtecan Eligibility criteria: Inclusion Criteria:  * Phase 1 (Part A): Patients must be at least 2 years and less than 12 years of age at the time of study enrollment * Phase 2: Wilms tumor patients (Part B1): All Wilms tumor patients e

In [12]:
print(
    "Documents created:",
    trials_30["retrieval_document"].notna().sum()
)

print(
    "Empty documents:",
    (
        trials_30["retrieval_document"]
        .str.strip()
        .eq("")
    ).sum()
)

Documents created: 30
Empty documents: 0


In [13]:
with open(
    PATIENTS_DIR / "synthetic_patients.json",
    "r",
    encoding="utf-8"
) as f:
    synthetic_patients = json.load(f)

print(
    "Synthetic patients loaded:",
    len(synthetic_patients)
)

Synthetic patients loaded: 8


In [14]:
patients_by_id = {
    patient["patient_id"]: patient
    for patient in synthetic_patients
}

P001 = patients_by_id["P001"]
P002 = patients_by_id["P002"]
P003 = patients_by_id["P003"]
P004 = patients_by_id["P004"]
P005 = patients_by_id["P005"]
P006 = patients_by_id["P006"]
P007 = patients_by_id["P007"]
P008 = patients_by_id["P008"]

In [15]:
def build_patient_query(patient):

    diagnosis = patient["diagnosis"]
    performance = patient["performance_status"]
    labs = patient["laboratory_values"]
    imaging = patient["surgical_imaging"]

    metastatic_sites = diagnosis.get(
        "metastatic_sites",
        []
    )

    query_parts = [
        f"Patient age {patient['age_years']} years.",
        f"Sex: {patient['sex']}.",
        f"Diagnosis: {diagnosis['grade']} {diagnosis['cancer_type']}.",
        f"Disease status: {diagnosis['disease_status']}.",
        f"Disease extent: {diagnosis['disease_extent']}.",
        f"Metastatic sites: {', '.join(metastatic_sites) if metastatic_sites else 'none'}.",
        f"Measurable disease: {diagnosis['measurable_disease']}."
    ]

    if performance["score"] is not None:
        query_parts.append(
            f"Performance status: "
            f"{performance['scale']} "
            f"{performance['score']}."
        )

    if labs["anc_per_ul"] is not None:
        query_parts.append(
            f"ANC {labs['anc_per_ul']} per uL."
        )

    if labs["platelets_per_ul"] is not None:
        query_parts.append(
            f"Platelets {labs['platelets_per_ul']} per uL."
        )

    if labs["gfr_ml_min_1_73m2"] is not None:
        query_parts.append(
            f"GFR {labs['gfr_ml_min_1_73m2']} "
            f"mL/min/1.73m2."
        )

    if imaging[
        "pulmonary_metastatic_disease_suspected"
    ] is True:
        query_parts.append(
            "Pulmonary metastatic disease is suspected."
        )

    if imaging[
        "largest_pulmonary_nodule_mm"
    ] is not None:
        query_parts.append(
            f"Largest pulmonary nodule "
            f"{imaging['largest_pulmonary_nodule_mm']} mm."
        )

    return " ".join(query_parts)

In [16]:
P003_query = build_patient_query(P003)

print(P003_query)

Patient age 17 years. Sex: Male. Diagnosis: high-grade osteosarcoma. Disease status: relapsed. Disease extent: metastatic. Metastatic sites: lung. Measurable disease: True. Performance status: Karnofsky 80. ANC 1420 per uL. Platelets 118000 per uL. GFR 88 mL/min/1.73m2.


In [17]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=10000
)

trial_matrix = tfidf_vectorizer.fit_transform(
    trials_30["retrieval_document"]
)

print(
    "Trial matrix shape:",
    trial_matrix.shape
)

Trial matrix shape: (30, 10000)


In [18]:
def retrieve_trials(
    patient,
    top_k=5
):

    patient_query = build_patient_query(
        patient
    )

    patient_vector = tfidf_vectorizer.transform(
        [patient_query]
    )

    similarities = cosine_similarity(
        patient_vector,
        trial_matrix
    )[0]

    ranked_indices = similarities.argsort()[::-1][
        :top_k
    ]

    results = []

    for rank, idx in enumerate(
        ranked_indices,
        start=1
    ):

        row = trials_30.iloc[idx]

        results.append({
            "rank": rank,
            "nct_id": row["NCT Number"],
            "title": row["Study Title"],
            "study_status": row["Study Status"],
            "similarity_score": float(
                similarities[idx]
            )
        })

    return pd.DataFrame(results)

In [19]:
retrieve_trials(
    P003,
    top_k=5
)

,rank,nct_id,title,study_status,similarity_score
0,1,NCT07321912,Eflornithine (DFMO) for Ewing Sarcoma and Oste...,RECRUITING,0.122701
1,2,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,ACTIVE_NOT_RECRUITING,0.096662
2,3,NCT06650514,A Phase 2 Pilot Study to Evaluate the Safety a...,RECRUITING,0.092572
3,4,NCT07477457,"A Study of Gefitinib, Trametinib, Disulfiram, ...",RECRUITING,0.088545
4,5,NCT06235125,Safety and Feasibility of Intraoperative Visua...,RECRUITING,0.085547


In [20]:
retrieve_trials(
    P001,
    top_k=5
)

,rank,nct_id,title,study_status,similarity_score
0,1,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,ACTIVE_NOT_RECRUITING,0.104054
1,2,NCT07321912,Eflornithine (DFMO) for Ewing Sarcoma and Oste...,RECRUITING,0.096069
2,3,NCT06650514,A Phase 2 Pilot Study to Evaluate the Safety a...,RECRUITING,0.090127
3,4,NCT07560826,The CATSINDO Trial - Clinical and Translationa...,RECRUITING,0.081534
4,5,NCT05590572,A Study of Sulfatinib on Relapsed or Refractor...,NOT_YET_RECRUITING,0.078352


In [21]:
retrieve_trials(
    P005,
    top_k=5
)

,rank,nct_id,title,study_status,similarity_score
0,1,NCT06650514,A Phase 2 Pilot Study to Evaluate the Safety a...,RECRUITING,0.110948
1,2,NCT07321912,Eflornithine (DFMO) for Ewing Sarcoma and Oste...,RECRUITING,0.106404
2,3,NCT07725380,"Domatinostat With Sirolimus for Relapsed, Refr...",RECRUITING,0.098099
3,4,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,ACTIVE_NOT_RECRUITING,0.097688
4,5,NCT07477457,"A Study of Gefitinib, Trametinib, Disulfiram, ...",RECRUITING,0.088667


In [22]:
retrieve_trials(
    P006,
    top_k=5
)

,rank,nct_id,title,study_status,similarity_score
0,1,NCT07321912,Eflornithine (DFMO) for Ewing Sarcoma and Oste...,RECRUITING,0.139619
1,2,NCT05235165,Thoracotomy Versus Thoracoscopic Management of...,RECRUITING,0.135914
2,3,NCT03628209,Nivolumab or Nivolumab and Azacitidine in Pati...,ACTIVE_NOT_RECRUITING,0.105499
3,4,NCT07477457,"A Study of Gefitinib, Trametinib, Disulfiram, ...",RECRUITING,0.095289
4,5,NCT06235125,Safety and Feasibility of Intraoperative Visua...,RECRUITING,0.087850


In [23]:
retrieve_trials(
    P007,
    top_k=5
)

,rank,nct_id,title,study_status,similarity_score
0,1,NCT07321912,Eflornithine (DFMO) for Ewing Sarcoma and Oste...,RECRUITING,0.106882
1,2,NCT07774533,68Ga-B7-H3 Nanobody ImmunoPET in Osteosarcoma,RECRUITING,0.103488
2,3,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,ACTIVE_NOT_RECRUITING,0.101806
3,4,NCT06650514,A Phase 2 Pilot Study to Evaluate the Safety a...,RECRUITING,0.098299
4,5,NCT05590572,A Study of Sulfatinib on Relapsed or Refractor...,NOT_YET_RECRUITING,0.089712


## Evaluate Baseline Retrieval

The five synthetic showcase patients were intentionally designed around five selected trials.
We use those known patient-trial pairs as a small retrieval benchmark.

In [24]:
expected_trial_by_patient = {
    "P001": "NCT07560826",
    "P003": "NCT07144254",
    "P005": "NCT05590572",
    "P006": "NCT06235125",
    "P007": "NCT02013336"
}

In [25]:
baseline_retrieval_results = []

for patient_id, expected_nct in expected_trial_by_patient.items():

    patient = patients_by_id[patient_id]

    retrieved = retrieve_trials(
        patient,
        top_k=5
    )

    retrieved_ids = retrieved["nct_id"].tolist()

    hit = expected_nct in retrieved_ids

    if hit:
        rank = retrieved_ids.index(expected_nct) + 1
    else:
        rank = None

    baseline_retrieval_results.append({
        "patient_id": patient_id,
        "expected_trial": expected_nct,
        "hit_at_5": hit,
        "rank": rank
    })

baseline_results_df = pd.DataFrame(
    baseline_retrieval_results
)

baseline_results_df

,patient_id,expected_trial,hit_at_5,rank
0,P001,NCT07560826,True,4.0
1,P003,NCT07144254,False,NaN
2,P005,NCT05590572,False,NaN
3,P006,NCT06235125,True,5.0
4,P007,NCT02013336,False,NaN


In [26]:
baseline_hit_at_5 = (
    baseline_results_df["hit_at_5"].mean()
)

print(
    f"Baseline Hit@5: {baseline_hit_at_5:.2f}"
)

print(
    f"Baseline Recall@5: {baseline_hit_at_5 * 100:.1f}%"
)

Baseline Hit@5: 0.40
Baseline Recall@5: 40.0%


In [27]:
def build_patient_query_v2(patient):

    diagnosis = patient["diagnosis"]
    prior = patient["prior_treatment"]
    current = patient["current_treatment"]
    timing = patient["treatment_timing_days"]
    labs = patient["laboratory_values"]
    cardiac = patient["cardiac"]
    pulmonary = patient["pulmonary"]
    clinical = patient["other_clinical_facts"]
    imaging = patient["surgical_imaging"]

    parts = []

    parts.append(
        f"{patient['age_years']} year old "
        f"{diagnosis['grade']} "
        f"{diagnosis['cancer_type']} patient."
    )

    parts.append(
        f"Disease is {diagnosis['disease_status']} "
        f"and {diagnosis['disease_extent']}."
    )

    if diagnosis["histologic_confirmation"] is True:
        parts.append(
            "Histologically confirmed osteosarcoma."
        )

    if diagnosis["measurable_disease"] is True:
        parts.append(
            "Measurable disease is present."
        )

    metastatic_sites = diagnosis.get(
        "metastatic_sites",
        []
    )

    if metastatic_sites:
        parts.append(
            "Metastatic disease sites include "
            + ", ".join(metastatic_sites)
            + "."
        )

    prior_drugs = []

    for drug in [
        "methotrexate",
        "doxorubicin",
        "cisplatin",
        "ifosfamide"
    ]:

        if prior.get(drug) is True:
            prior_drugs.append(drug)

    if prior_drugs:
        parts.append(
            "Prior anticancer treatment included "
            + ", ".join(prior_drugs)
            + "."
        )

    if current.get("map_chemotherapy") is True:
        parts.append(
            "Currently receiving or planning MAP chemotherapy."
        )

    if current.get("methotrexate_dose_g_m2") is not None:
        parts.append(
            f"Methotrexate dose "
            f"{current['methotrexate_dose_g_m2']} g/m2."
        )

    if timing["last_chemotherapy"] is not None:
        parts.append(
            f"Last chemotherapy was "
            f"{timing['last_chemotherapy']} days ago."
        )

    if timing["last_major_surgery"] is not None:
        parts.append(
            f"Last major surgery was "
            f"{timing['last_major_surgery']} days ago."
        )

    if clinical["life_expectancy_months"] is not None:
        parts.append(
            f"Life expectancy approximately "
            f"{clinical['life_expectancy_months']} months."
        )

    if clinical["blood_pressure_controlled"] is not None:
        parts.append(
            f"Blood pressure controlled: "
            f"{clinical['blood_pressure_controlled']}."
        )

    if labs["anc_per_ul"] is not None:
        parts.append(
            f"ANC {labs['anc_per_ul']} per uL."
        )

    if labs["platelets_per_ul"] is not None:
        parts.append(
            f"Platelets {labs['platelets_per_ul']} per uL."
        )

    if labs["hemoglobin_g_dl"] is not None:
        parts.append(
            f"Hemoglobin {labs['hemoglobin_g_dl']} g/dL."
        )

    if labs["gfr_ml_min_1_73m2"] is not None:
        parts.append(
            f"GFR {labs['gfr_ml_min_1_73m2']} "
            f"mL/min/1.73m2."
        )

    if cardiac["lvef_percent"] is not None:
        parts.append(
            f"LVEF {cardiac['lvef_percent']} percent."
        )

    if cardiac["qtc_ms"] is not None:
        parts.append(
            f"QTc {cardiac['qtc_ms']} milliseconds."
        )

    if pulmonary["oxygen_saturation_percent"] is not None:
        parts.append(
            f"Room air oxygen saturation "
            f"{pulmonary['oxygen_saturation_percent']} percent."
        )

    if imaging[
        "pulmonary_metastatic_disease_suspected"
    ] is True:
        parts.append(
            "Imaging suspicious for pulmonary metastatic disease."
        )

    if imaging[
        "largest_pulmonary_nodule_mm"
    ] is not None:
        parts.append(
            f"Pulmonary nodule measures "
            f"{imaging['largest_pulmonary_nodule_mm']} mm."
        )

    if imaging[
        "pulmonary_surgery_recommended"
    ] is True:
        parts.append(
            "Pulmonary surgery is recommended."
        )

    return " ".join(parts)

In [28]:
print(
    build_patient_query_v2(P003)
)

17 year old high-grade osteosarcoma patient. Disease is relapsed and metastatic. Histologically confirmed osteosarcoma. Measurable disease is present. Metastatic disease sites include lung. Prior anticancer treatment included methotrexate, doxorubicin, cisplatin. Last chemotherapy was 28 days ago. Last major surgery was 35 days ago. ANC 1420 per uL. Platelets 118000 per uL. GFR 88 mL/min/1.73m2. QTc 442 milliseconds. Room air oxygen saturation 97 percent.


In [29]:
def retrieve_trials_v2(
    patient,
    top_k=5
):

    patient_query = build_patient_query_v2(
        patient
    )

    patient_vector = tfidf_vectorizer.transform(
        [patient_query]
    )

    similarities = cosine_similarity(
        patient_vector,
        trial_matrix
    )[0]

    ranked_indices = similarities.argsort()[::-1][
        :top_k
    ]

    results = []

    for rank, idx in enumerate(
        ranked_indices,
        start=1
    ):

        row = trials_30.iloc[idx]

        results.append({
            "rank": rank,
            "nct_id": row["NCT Number"],
            "title": row["Study Title"],
            "study_status": row["Study Status"],
            "similarity_score": float(
                similarities[idx]
            )
        })

    return pd.DataFrame(results)

In [30]:
v2_retrieval_results = []

for patient_id, expected_nct in expected_trial_by_patient.items():

    patient = patients_by_id[patient_id]

    retrieved = retrieve_trials_v2(
        patient,
        top_k=5
    )

    retrieved_ids = retrieved["nct_id"].tolist()

    hit = expected_nct in retrieved_ids

    rank = (
        retrieved_ids.index(expected_nct) + 1
        if hit
        else None
    )

    v2_retrieval_results.append({
        "patient_id": patient_id,
        "expected_trial": expected_nct,
        "hit_at_5": hit,
        "rank": rank
    })

v2_results_df = pd.DataFrame(
    v2_retrieval_results
)

v2_results_df

,patient_id,expected_trial,hit_at_5,rank
0,P001,NCT07560826,True,1.0
1,P003,NCT07144254,True,3.0
2,P005,NCT05590572,True,2.0
3,P006,NCT06235125,True,5.0
4,P007,NCT02013336,False,NaN


In [31]:
v2_hit_at_5 = (
    v2_results_df["hit_at_5"].mean()
)

print(
    f"Baseline Hit@5: "
    f"{baseline_hit_at_5:.2f}"
)

print(
    f"Improved TF-IDF Hit@5: "
    f"{v2_hit_at_5:.2f}"
)

Baseline Hit@5: 0.40
Improved TF-IDF Hit@5: 0.80


In [32]:
%pip install sentence-transformers

  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/740.6 kB ? eta -:--:--
   ---------------------------- ----------- 524.3/740.6 kB 3.1 MB/s eta 0:00:01
   ---------------------------------------- 740.6/740.6 kB 3.1 MB/s  0:00:00
   ---------------------------------------- 0.0/846.4 kB ? eta -:--:--
   ---------------------------------------- 846.4/846.4 kB 4.0 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ------------- -------------------------- 1.3/3.8 MB 6.4 MB/s eta 0:00:01
   ------------------------------ --------- 2.9/3.8 MB 7.3 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 7.4 MB/s  0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ------- -------------------------------- 2.4/12.6 MB 11.9 MB/s eta 0:00:01
   --------------- ------------------------ 5.0/12.6 

In [34]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [35]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

C:\Users\Sista\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sista\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\Sista\AppData\Local\Programs\Python\Python313\Lib\site-packages\hu

Embedding model loaded successfully.


In [36]:
trial_embeddings = embedding_model.encode(
    trials_30["retrieval_document"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    "Trial embedding shape:",
    trial_embeddings.shape
)

Trial embedding shape: (30, 384)


In [37]:
def retrieve_trials_embeddings(
    patient,
    top_k=5
):

    patient_query = build_patient_query_v2(
        patient
    )

    patient_embedding = embedding_model.encode(
        [patient_query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    similarities = np.dot(
        trial_embeddings,
        patient_embedding
    )

    ranked_indices = similarities.argsort()[::-1][
        :top_k
    ]

    results = []

    for rank, idx in enumerate(
        ranked_indices,
        start=1
    ):

        row = trials_30.iloc[idx]

        results.append({
            "rank": rank,
            "nct_id": row["NCT Number"],
            "title": row["Study Title"],
            "study_status": row["Study Status"],
            "similarity_score": float(
                similarities[idx]
            )
        })

    return pd.DataFrame(results)

In [38]:
retrieve_trials_embeddings(
    P001,
    top_k=5
)

,rank,nct_id,title,study_status,similarity_score
0,1,NCT07560826,The CATSINDO Trial - Clinical and Translationa...,RECRUITING,0.720470
1,2,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,ACTIVE_NOT_RECRUITING,0.700234
2,3,NCT07428993,Evaluating Efficacy of B7-H3-CAR T Cells Admin...,RECRUITING,0.688136
3,4,NCT06650514,A Phase 2 Pilot Study to Evaluate the Safety a...,RECRUITING,0.674088
4,5,NCT06714383,Trilaciclib Combing Chemotherapy in the Neoadj...,RECRUITING,0.665622


In [39]:
retrieve_trials_embeddings(
    P003,
    top_k=5
)

,rank,nct_id,title,study_status,similarity_score
0,1,NCT07428993,Evaluating Efficacy of B7-H3-CAR T Cells Admin...,RECRUITING,0.777525
1,2,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,ACTIVE_NOT_RECRUITING,0.762698
2,3,NCT07479732,Apatinib Combined With Liposomal Irinotecan fo...,RECRUITING,0.736265
3,4,NCT06650514,A Phase 2 Pilot Study to Evaluate the Safety a...,RECRUITING,0.727781
4,5,NCT07477457,"A Study of Gefitinib, Trametinib, Disulfiram, ...",RECRUITING,0.717470


In [40]:
retrieve_trials_embeddings(
    P005,
    top_k=5
)

,rank,nct_id,title,study_status,similarity_score
0,1,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,ACTIVE_NOT_RECRUITING,0.744667
1,2,NCT07428993,Evaluating Efficacy of B7-H3-CAR T Cells Admin...,RECRUITING,0.715985
2,3,NCT07560826,The CATSINDO Trial - Clinical and Translationa...,RECRUITING,0.704461
3,4,NCT07479732,Apatinib Combined With Liposomal Irinotecan fo...,RECRUITING,0.701954
4,5,NCT05590572,A Study of Sulfatinib on Relapsed or Refractor...,NOT_YET_RECRUITING,0.698555


In [41]:
retrieve_trials_embeddings(
    P006,
    top_k=5
)

,rank,nct_id,title,study_status,similarity_score
0,1,NCT06114225,Pulmonary Resectable Osteosarcoma Treated by M...,RECRUITING,0.703267
1,2,NCT07477457,"A Study of Gefitinib, Trametinib, Disulfiram, ...",RECRUITING,0.697032
2,3,NCT07428993,Evaluating Efficacy of B7-H3-CAR T Cells Admin...,RECRUITING,0.691345
3,4,NCT03628209,Nivolumab or Nivolumab and Azacitidine in Pati...,ACTIVE_NOT_RECRUITING,0.686059
4,5,NCT05235165,Thoracotomy Versus Thoracoscopic Management of...,RECRUITING,0.678687


In [42]:
retrieve_trials_embeddings(
    P007,
    top_k=5
)

,rank,nct_id,title,study_status,similarity_score
0,1,NCT07428993,Evaluating Efficacy of B7-H3-CAR T Cells Admin...,RECRUITING,0.754811
1,2,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,ACTIVE_NOT_RECRUITING,0.745687
2,3,NCT07560826,The CATSINDO Trial - Clinical and Translationa...,RECRUITING,0.724184
3,4,NCT07479732,Apatinib Combined With Liposomal Irinotecan fo...,RECRUITING,0.715444
4,5,NCT06650514,A Phase 2 Pilot Study to Evaluate the Safety a...,RECRUITING,0.711360


In [43]:
embedding_retrieval_results = []

for patient_id, expected_nct in expected_trial_by_patient.items():

    patient = patients_by_id[patient_id]

    retrieved = retrieve_trials_embeddings(
        patient,
        top_k=5
    )

    retrieved_ids = retrieved[
        "nct_id"
    ].tolist()

    hit = expected_nct in retrieved_ids

    rank = (
        retrieved_ids.index(expected_nct) + 1
        if hit
        else None
    )

    embedding_retrieval_results.append({
        "patient_id": patient_id,
        "expected_trial": expected_nct,
        "hit_at_5": hit,
        "rank": rank
    })

embedding_results_df = pd.DataFrame(
    embedding_retrieval_results
)

embedding_results_df

,patient_id,expected_trial,hit_at_5,rank
0,P001,NCT07560826,True,1.0
1,P003,NCT07144254,False,NaN
2,P005,NCT05590572,True,5.0
3,P006,NCT06235125,False,NaN
4,P007,NCT02013336,False,NaN


In [44]:
embedding_hit_at_5 = (
    embedding_results_df[
        "hit_at_5"
    ].mean()
)

print(
    f"Embedding Hit@5: "
    f"{embedding_hit_at_5:.2f}"
)

Embedding Hit@5: 0.40


In [45]:
print(
    f"Baseline TF-IDF Hit@5: "
    f"{baseline_hit_at_5:.2f}"
)

print(
    f"Improved TF-IDF Hit@5: "
    f"{v2_hit_at_5:.2f}"
)

print(
    f"Embedding Retrieval Hit@5: "
    f"{embedding_hit_at_5:.2f}"
)

Baseline TF-IDF Hit@5: 0.40
Improved TF-IDF Hit@5: 0.80
Embedding Retrieval Hit@5: 0.40


In [46]:
comparison_df = (
    baseline_results_df[
        ["patient_id", "rank"]
    ]
    .rename(
        columns={
            "rank": "baseline_rank"
        }
    )
    .merge(
        v2_results_df[
            ["patient_id", "rank"]
        ].rename(
            columns={
                "rank": "tfidf_v2_rank"
            }
        ),
        on="patient_id"
    )
    .merge(
        embedding_results_df[
            ["patient_id", "rank"]
        ].rename(
            columns={
                "rank": "embedding_rank"
            }
        ),
        on="patient_id"
    )
)

comparison_df

,patient_id,baseline_rank,tfidf_v2_rank,embedding_rank
0,P001,4.0,1.0,1.0
1,P003,NaN,3.0,NaN
2,P005,NaN,2.0,5.0
3,P006,5.0,5.0,NaN
4,P007,NaN,NaN,NaN


In [47]:
print("TF-IDF V2 Top 10 for P007:")
display(
    retrieve_trials_v2(
        P007,
        top_k=10
    )
)

print("\nEmbedding Top 10 for P007:")
display(
    retrieve_trials_embeddings(
        P007,
        top_k=10
    )
)

TF-IDF V2 Top 10 for P007:


,rank,nct_id,title,study_status,similarity_score
0,1,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,ACTIVE_NOT_RECRUITING,0.130996
1,2,NCT03900793,Losartan + Sunitinib in Treatment of Osteosarcoma,RECRUITING,0.125319
2,3,NCT05019703,Atezolizumab and Cabozantinib for the Treatmen...,RECRUITING,0.120560
3,4,NCT07321912,Eflornithine (DFMO) for Ewing Sarcoma and Oste...,RECRUITING,0.115749
4,5,NCT07725380,"Domatinostat With Sirolimus for Relapsed, Refr...",RECRUITING,0.104647
5,6,NCT04616560,Trastuzumab Deruxtecan (DS-8201a) for the Trea...,RECRUITING,0.103933
6,7,NCT07144254,Tegavivint With Gemcitabine in Patients With R...,RECRUITING,0.101830
7,8,NCT07477457,"A Study of Gefitinib, Trametinib, Disulfiram, ...",RECRUITING,0.094416
8,9,NCT05590572,A Study of Sulfatinib on Relapsed or Refractor...,NOT_YET_RECRUITING,0.092233
9,10,NCT06650514,A Phase 2 Pilot Study to Evaluate the Safety a...,RECRUITING,0.090369



Embedding Top 10 for P007:


,rank,nct_id,title,study_status,similarity_score
0,1,NCT07428993,Evaluating Efficacy of B7-H3-CAR T Cells Admin...,RECRUITING,0.754811
1,2,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,ACTIVE_NOT_RECRUITING,0.745687
2,3,NCT07560826,The CATSINDO Trial - Clinical and Translationa...,RECRUITING,0.724184
3,4,NCT07479732,Apatinib Combined With Liposomal Irinotecan fo...,RECRUITING,0.715444
4,5,NCT06650514,A Phase 2 Pilot Study to Evaluate the Safety a...,RECRUITING,0.711360
5,6,NCT07477457,"A Study of Gefitinib, Trametinib, Disulfiram, ...",RECRUITING,0.709465
6,7,NCT06714383,Trilaciclib Combing Chemotherapy in the Neoadj...,RECRUITING,0.696813
7,8,NCT05590572,A Study of Sulfatinib on Relapsed or Refractor...,NOT_YET_RECRUITING,0.695701
8,9,NCT05019703,Atezolizumab and Cabozantinib for the Treatmen...,RECRUITING,0.680234
9,10,NCT07787429,Evaluation of the Safety and Efficacy of Mifam...,RECRUITING,0.676415


In [48]:
def build_patient_query_v3(patient):

    diagnosis = patient["diagnosis"]
    prior = patient["prior_treatment"]
    timing = patient["treatment_timing_days"]
    performance = patient["performance_status"]
    labs = patient["laboratory_values"]
    cardiac = patient["cardiac"]
    pulmonary = patient["pulmonary"]
    clinical = patient["other_clinical_facts"]
    imaging = patient["surgical_imaging"]

    parts = []

    parts.append(
        f"Pediatric osteosarcoma patient age "
        f"{patient['age_years']} years."
    )

    parts.append(
        f"Histologically confirmed "
        f"{diagnosis['grade']} osteosarcoma."
    )

    parts.append(
        f"Disease status is "
        f"{diagnosis['disease_status']}."
    )

    parts.append(
        f"Disease extent is "
        f"{diagnosis['disease_extent']}."
    )

    if diagnosis["disease_status"] in [
        "recurrent",
        "relapsed",
        "refractory"
    ]:
        parts.append(
            "Disease progression occurred after prior anticancer therapy."
        )

        parts.append(
            "Patient has recurrent or refractory pediatric solid tumor disease."
        )

    if diagnosis["measurable_disease"] is True:
        parts.append(
            "Measurable or evaluable disease is present."
        )

    metastatic_sites = diagnosis.get(
        "metastatic_sites",
        []
    )

    if metastatic_sites:
        parts.append(
            "Metastatic disease involves "
            + ", ".join(metastatic_sites)
            + "."
        )

    prior_drugs = [
        drug
        for drug in [
            "methotrexate",
            "doxorubicin",
            "cisplatin",
            "ifosfamide"
        ]
        if prior.get(drug) is True
    ]

    if prior_drugs:
        parts.append(
            "Previously treated with systemic anticancer therapy including "
            + ", ".join(prior_drugs)
            + "."
        )

    if prior.get("major_surgery") is True:
        parts.append(
            "Patient previously underwent major cancer surgery."
        )

    if timing["last_chemotherapy"] is not None:
        parts.append(
            f"Last chemotherapy was "
            f"{timing['last_chemotherapy']} days ago."
        )

    if timing["last_major_surgery"] is not None:
        parts.append(
            f"Last major surgery was "
            f"{timing['last_major_surgery']} days ago."
        )

    if performance["score"] is not None:
        parts.append(
            f"Performance status "
            f"{performance['scale']} "
            f"{performance['score']}."
        )

    if labs["anc_per_ul"] is not None:
        parts.append(
            f"ANC {labs['anc_per_ul']} per uL."
        )

    if labs["platelets_per_ul"] is not None:
        parts.append(
            f"Platelets {labs['platelets_per_ul']} per uL."
        )

    if labs["gfr_ml_min_1_73m2"] is not None:
        parts.append(
            f"Renal function GFR "
            f"{labs['gfr_ml_min_1_73m2']} "
            f"mL/min/1.73m2."
        )

    if cardiac["lvef_percent"] is not None:
        parts.append(
            f"LVEF {cardiac['lvef_percent']} percent."
        )

    if clinical["active_infection"] is False:
        parts.append(
            "No active infection."
        )

    if clinical["pregnant"] is False:
        parts.append(
            "Patient is not pregnant."
        )

    return " ".join(parts)

In [49]:
P007_query_v3 = build_patient_query_v3(P007)

print(P007_query_v3)

Pediatric osteosarcoma patient age 10 years. Histologically confirmed high-grade osteosarcoma. Disease status is recurrent. Disease extent is metastatic. Disease progression occurred after prior anticancer therapy. Patient has recurrent or refractory pediatric solid tumor disease. Measurable or evaluable disease is present. Metastatic disease involves lung. Previously treated with systemic anticancer therapy including methotrexate, doxorubicin, cisplatin. Patient previously underwent major cancer surgery. Last chemotherapy was 35 days ago. Last major surgery was 42 days ago. Performance status Lansky 80. ANC 1600 per uL. Platelets 135000 per uL. Renal function GFR 90 mL/min/1.73m2. LVEF 58 percent. No active infection. Patient is not pregnant.


In [50]:
def retrieve_trials_v3(
    patient,
    top_k=5
):

    patient_query = build_patient_query_v3(
        patient
    )

    patient_vector = tfidf_vectorizer.transform(
        [patient_query]
    )

    similarities = cosine_similarity(
        patient_vector,
        trial_matrix
    )[0]

    ranked_indices = similarities.argsort()[::-1][
        :top_k
    ]

    results = []

    for rank, idx in enumerate(
        ranked_indices,
        start=1
    ):

        row = trials_30.iloc[idx]

        results.append({
            "rank": rank,
            "nct_id": row["NCT Number"],
            "title": row["Study Title"],
            "study_status": row["Study Status"],
            "similarity_score": float(
                similarities[idx]
            )
        })

    return pd.DataFrame(results)

In [51]:
retrieve_trials_v3(
    P007,
    top_k=10
)

,rank,nct_id,title,study_status,similarity_score
0,1,NCT07725380,"Domatinostat With Sirolimus for Relapsed, Refr...",RECRUITING,0.143276
1,2,NCT02013336,Phase 1 Dose-escalating Study of MM-398 (Irino...,RECRUITING,0.138003
2,3,NCT03900793,Losartan + Sunitinib in Treatment of Osteosarcoma,RECRUITING,0.133860
3,4,NCT05590572,A Study of Sulfatinib on Relapsed or Refractor...,NOT_YET_RECRUITING,0.127839
4,5,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,ACTIVE_NOT_RECRUITING,0.125652
5,6,NCT07144254,Tegavivint With Gemcitabine in Patients With R...,RECRUITING,0.122426
6,7,NCT07321912,Eflornithine (DFMO) for Ewing Sarcoma and Oste...,RECRUITING,0.119085
7,8,NCT03628209,Nivolumab or Nivolumab and Azacitidine in Pati...,ACTIVE_NOT_RECRUITING,0.117545
8,9,NCT07774533,68Ga-B7-H3 Nanobody ImmunoPET in Osteosarcoma,RECRUITING,0.115012
9,10,NCT05019703,Atezolizumab and Cabozantinib for the Treatmen...,RECRUITING,0.112567


In [52]:
v3_retrieval_results = []

for patient_id, expected_nct in expected_trial_by_patient.items():

    patient = patients_by_id[
        patient_id
    ]

    retrieved = retrieve_trials_v3(
        patient,
        top_k=5
    )

    retrieved_ids = retrieved[
        "nct_id"
    ].tolist()

    hit = expected_nct in retrieved_ids

    rank = (
        retrieved_ids.index(expected_nct) + 1
        if hit
        else None
    )

    v3_retrieval_results.append({
        "patient_id": patient_id,
        "expected_trial": expected_nct,
        "hit_at_5": hit,
        "rank": rank
    })

v3_results_df = pd.DataFrame(
    v3_retrieval_results
)

v3_results_df

,patient_id,expected_trial,hit_at_5,rank
0,P001,NCT07560826,False,NaN
1,P003,NCT07144254,True,4.0
2,P005,NCT05590572,True,4.0
3,P006,NCT06235125,False,NaN
4,P007,NCT02013336,True,2.0


In [53]:
v3_hit_at_5 = (
    v3_results_df[
        "hit_at_5"
    ].mean()
)

print(
    f"Baseline TF-IDF Hit@5: "
    f"{baseline_hit_at_5:.2f}"
)

print(
    f"TF-IDF V2 Hit@5: "
    f"{v2_hit_at_5:.2f}"
)

print(
    f"TF-IDF V3 Hit@5: "
    f"{v3_hit_at_5:.2f}"
)

print(
    f"Embedding Hit@5: "
    f"{embedding_hit_at_5:.2f}"
)

Baseline TF-IDF Hit@5: 0.40
TF-IDF V2 Hit@5: 0.80
TF-IDF V3 Hit@5: 0.60
Embedding Hit@5: 0.40


V2 + V3 fused retriever

In [54]:
def retrieve_trials_fused(
    patient,
    top_k=5,
    v2_weight=0.7,
    v3_weight=0.3
):

    query_v2 = build_patient_query_v2(patient)
    query_v3 = build_patient_query_v3(patient)

    vector_v2 = tfidf_vectorizer.transform(
        [query_v2]
    )

    vector_v3 = tfidf_vectorizer.transform(
        [query_v3]
    )

    similarity_v2 = cosine_similarity(
        vector_v2,
        trial_matrix
    )[0]

    similarity_v3 = cosine_similarity(
        vector_v3,
        trial_matrix
    )[0]

    fused_scores = (
        v2_weight * similarity_v2
        +
        v3_weight * similarity_v3
    )

    ranked_indices = fused_scores.argsort()[::-1][
        :top_k
    ]

    results = []

    for rank, idx in enumerate(
        ranked_indices,
        start=1
    ):

        row = trials_30.iloc[idx]

        results.append({
            "rank": rank,
            "nct_id": row["NCT Number"],
            "title": row["Study Title"],
            "study_status": row["Study Status"],
            "v2_score": float(
                similarity_v2[idx]
            ),
            "v3_score": float(
                similarity_v3[idx]
            ),
            "fused_score": float(
                fused_scores[idx]
            )
        })

    return pd.DataFrame(results)

In [55]:
retrieve_trials_fused(
    P007,
    top_k=10
)

,rank,nct_id,title,study_status,v2_score,v3_score,fused_score
0,1,NCT03643133,Mifamurtide Combined With Post-operative Chemo...,ACTIVE_NOT_RECRUITING,0.130996,0.125652,0.129393
1,2,NCT03900793,Losartan + Sunitinib in Treatment of Osteosarcoma,RECRUITING,0.125319,0.133860,0.127881
2,3,NCT05019703,Atezolizumab and Cabozantinib for the Treatmen...,RECRUITING,0.120560,0.112567,0.118162
3,4,NCT07321912,Eflornithine (DFMO) for Ewing Sarcoma and Oste...,RECRUITING,0.115749,0.119085,0.116750
4,5,NCT07725380,"Domatinostat With Sirolimus for Relapsed, Refr...",RECRUITING,0.104647,0.143276,0.116236
5,6,NCT07144254,Tegavivint With Gemcitabine in Patients With R...,RECRUITING,0.101830,0.122426,0.108008
6,7,NCT04616560,Trastuzumab Deruxtecan (DS-8201a) for the Trea...,RECRUITING,0.103933,0.108065,0.105173
7,8,NCT05590572,A Study of Sulfatinib on Relapsed or Refractor...,NOT_YET_RECRUITING,0.092233,0.127839,0.102915
8,9,NCT05235165,Thoracotomy Versus Thoracoscopic Management of...,RECRUITING,0.089285,0.112313,0.096194
9,10,NCT07477457,"A Study of Gefitinib, Trametinib, Disulfiram, ...",RECRUITING,0.094416,0.099411,0.095914


In [56]:
fused_retrieval_results = []

for patient_id, expected_nct in expected_trial_by_patient.items():

    patient = patients_by_id[patient_id]

    retrieved = retrieve_trials_fused(
        patient,
        top_k=5
    )

    retrieved_ids = retrieved["nct_id"].tolist()

    hit = expected_nct in retrieved_ids

    rank = (
        retrieved_ids.index(expected_nct) + 1
        if hit
        else None
    )

    fused_retrieval_results.append({
        "patient_id": patient_id,
        "expected_trial": expected_nct,
        "hit_at_5": hit,
        "rank": rank
    })

fused_results_df = pd.DataFrame(
    fused_retrieval_results
)

fused_results_df

,patient_id,expected_trial,hit_at_5,rank
0,P001,NCT07560826,True,1.0
1,P003,NCT07144254,True,2.0
2,P005,NCT05590572,True,3.0
3,P006,NCT06235125,False,NaN
4,P007,NCT02013336,False,NaN


In [57]:
fused_hit_at_5 = fused_results_df[
    "hit_at_5"
].mean()

print(
    f"Fused TF-IDF Hit@5: "
    f"{fused_hit_at_5:.2f}"
)

Fused TF-IDF Hit@5: 0.60


In [58]:
retrieval_comparison = pd.DataFrame({
    "method": [
        "Baseline TF-IDF",
        "TF-IDF V2",
        "TF-IDF V3",
        "Sentence Embeddings",
        "V2 + V3 Fusion"
    ],
    "hit_at_5": [
        baseline_hit_at_5,
        v2_hit_at_5,
        v3_hit_at_5,
        embedding_hit_at_5,
        fused_hit_at_5
    ]
})

retrieval_comparison

,method,hit_at_5
0,Baseline TF-IDF,0.4
1,TF-IDF V2,0.8
2,TF-IDF V3,0.6
3,Sentence Embeddings,0.4
4,V2 + V3 Fusion,0.6


In [59]:
final_comparison_df = (
    baseline_results_df[
        ["patient_id", "rank"]
    ]
    .rename(columns={"rank": "baseline_rank"})
    .merge(
        v2_results_df[
            ["patient_id", "rank"]
        ].rename(columns={"rank": "v2_rank"}),
        on="patient_id"
    )
    .merge(
        v3_results_df[
            ["patient_id", "rank"]
        ].rename(columns={"rank": "v3_rank"}),
        on="patient_id"
    )
    .merge(
        embedding_results_df[
            ["patient_id", "rank"]
        ].rename(columns={"rank": "embedding_rank"}),
        on="patient_id"
    )
    .merge(
        fused_results_df[
            ["patient_id", "rank"]
        ].rename(columns={"rank": "fused_rank"}),
        on="patient_id"
    )
)

final_comparison_df

,patient_id,baseline_rank,v2_rank,v3_rank,embedding_rank,fused_rank
0,P001,4.0,1.0,NaN,1.0,1.0
1,P003,NaN,3.0,4.0,NaN,2.0
2,P005,NaN,2.0,4.0,5.0,3.0
3,P006,5.0,5.0,NaN,NaN,NaN
4,P007,NaN,NaN,2.0,NaN,NaN
